In [1]:
## Cell 1 · Imports

import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

df = pd.read_csv("csv/00_base_data.csv")
print(f"Loaded {len(df)} records")

PEDESTRIAN_CSV = "../ramy/pedestrian_mobility/Pedestrian_Mobility_Plan_Pedestrian_Demand_20260507.csv"

Loaded 2915 records


In [2]:
## Cell 2 · Check Pedestrian Data

ped = pd.read_csv(PEDESTRIAN_CSV)
print(f"Shape: {ped.shape}")
print(f"Columns: {list(ped.columns)}")
print(ped.head(3))

Shape: (127277, 20)
Columns: ['the_geom', 'BoroCode', 'BoroName', 'BoroCD', 'CounDist', 'AssemDist', 'StSenDist', 'CongDist', 'street', 'segmentid', 'Rank', 'PMP_ID', 'NTA2020', 'Boro', 'Category', 'NTAName', 'FEMAFldz', 'FEMAFldT', 'HrcEvac', 'SHAPE_Leng']
                                            the_geom BoroCode       BoroName  \
0  MULTILINESTRING ((-73.81665096623519 40.674537...        4         Queens   
1  MULTILINESTRING ((-74.25422309679644 40.506165...        5  Staten Island   
2  MULTILINESTRING ((-73.82885385076081 40.698186...        4         Queens   

  BoroCD CounDist AssemDist StSenDist CongDist              street  segmentid  \
0    410    28,32        31        10        5          122 STREET      55131   
1    503       51        62        24       11  WARDS POINT AVENUE         16   
2    409       29        24        14        5          120 STREET      56031   

   Rank  PMP_ID NTA2020 Boro   Category                 NTAName FEMAFldz  \
0     4   52434  QN1

In [3]:
## Cell 3 · Check Category and Rank values

print("Category distribution:")
print(ped["Category"].value_counts())
print("\nRank distribution:")
print(ped["Rank"].value_counts().sort_index())

Category distribution:
Category
Baseline        64975
Community       34195
Neighborhood    22769
Regional         4487
Global            851
Name: count, dtype: int64

Rank distribution:
Rank
1      851
2     4487
3    22769
4    34195
5    64975
Name: count, dtype: int64


In [4]:
## Cell 4 · Extract Coordinates & Match

import re

def extract_centroid(geom_str):
    try:
        coords = re.findall(r'(-?\d+\.\d+)\s+(-?\d+\.\d+)', geom_str)
        lons = [float(c[0]) for c in coords]
        lats = [float(c[1]) for c in coords]
        return np.mean(lats), np.mean(lons)
    except Exception:
        return None, None

print("Extracting centroids from geometry...")
ped["lat_c"] = None
ped["lon_c"] = None
ped[["lat_c", "lon_c"]] = ped["the_geom"].apply(
    lambda g: pd.Series(extract_centroid(g))
)
ped_clean = ped.dropna(subset=["lat_c", "lon_c"]).copy()
print(f"✓ {len(ped_clean)} segments with coordinates")

# Build spatial index
ped_coords = np.radians(ped_clean[["lat_c", "lon_c"]].values)
tree = BallTree(ped_coords, metric="haversine")

# Match each shop to nearest street segment
shop_coords = np.radians(df[["lat", "lon"]].values)
distances, indices = tree.query(shop_coords, k=1)

df["pedestrian_rank"]     = ped_clean.iloc[indices.flatten()]["Rank"].values
df["pedestrian_category"] = ped_clean.iloc[indices.flatten()]["Category"].values

print(f"\npedestrian_rank fill     : {df['pedestrian_rank'].notna().sum()}/{len(df)}")
print(f"pedestrian_category fill : {df['pedestrian_category'].notna().sum()}/{len(df)}")
print(f"\nDistribution:")
print(df["pedestrian_category"].value_counts())

Extracting centroids from geometry...
✓ 127277 segments with coordinates

pedestrian_rank fill     : 2915/2915
pedestrian_category fill : 2915/2915

Distribution:
pedestrian_category
Global          1376
Regional        1210
Neighborhood     297
Community         29
Baseline           3
Name: count, dtype: int64


In [6]:
category_map = {
    "Global"       : "highest",
    "Regional"     : "high",
    "Neighborhood" : "medium",
    "Community"    : "low",
    "Baseline"     : "lowest"
}

df["pedestrian_category"] = df["pedestrian_category"].map(category_map)
print(df["pedestrian_category"].value_counts())

pedestrian_category
highest    1376
high       1210
medium      297
low          29
lowest        3
Name: count, dtype: int64


In [7]:
## Cell 5 · Save

df_out = df[["osm_id", "pedestrian_rank", "pedestrian_category"]]
df_out.to_csv("csv/09_foot_traffic.csv", index=False, encoding="utf-8")
print(f"Saved {len(df_out)} records to csv/09_foot_traffic.csv")

Saved 2915 records to csv/09_foot_traffic.csv
